In [21]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

In [22]:
num_classes = 10

In [23]:
(x_train, y_train), (x_test, y_test) = datasets.fashion_mnist.load_data()

### ResNet blocks

In [24]:
def res_block(x, filters, stride = 1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides = stride, padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, strides = 1, padding = 'same')(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides = stride, padding = 'same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut]) # x.shape == shortcut.shape 이어야 함
    x = layers.ReLU()(x)

    return x

In [25]:
def myModel(input_shape, filters, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)

    net = layers.Conv2D(filters, 3, 1, padding='same')(inputs)
    net = layers.BatchNormalization()(net)
    net = layers.ReLU()(net)

    net = res_block(net, 64)
    net = res_block(net, 64)
    net = res_block(net, 128, stride = 2) # half size
    net = res_block(net, 128)
    net = res_block(net, 256, stride = 2) # half size
    net = res_block(net, 256)

    net = layers.GlobalAveragePooling2D()(net)

    net = tf.keras.layers.Dense(num_classes, activation='softmax')(net)

    model = tf.keras.Model(inputs=inputs, outputs=net, name='test')

    return(model)

In [26]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

In [27]:
x_train_4d = x_train / 255.0 # scaling to [0,1]
x_test_4d = x_test / 255.0 # scaling to [0,1]

In [28]:
x_train_4d.shape, x_test_4d.shape

((50000, 32, 32, 3), (10000, 32, 32, 3))

In [29]:
input_shape = (32, 32, 3)
filters =32
model = myModel(input_shape, filters, num_classes)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss = 'sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [ ]:
num_epochs = 10
bs = 64
hist = model.fit(x_train_4d, y_train, validation_data = (x_test_4d, y_test), batch_size=bs, shuffle=True, epochs=num_epochs)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 134s 165ms/step - accuracy: 0.5588 - loss: 1.2260 - val_accuracy: 0.5147 - val_loss: 1.6093
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 139s 178ms/step - accuracy: 0.7230 - loss: 0.7788 - val_accuracy: 0.7064 - val_loss: 0.8569
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 141s 180ms/step - accuracy: 0.7924 - loss: 0.5940 - val_accuracy: 0.6707 - val_loss: 0.9421
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 141s 180ms/step - accuracy: 0.8376 - loss: 0.4723 - val_accuracy: 0.7562 - val_loss: 0.7261
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 145s 185ms/step - accuracy: 0.8646 - loss: 0.3899 - val_accuracy: 0.7802 - val_loss: 0.7017
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 145s 185ms/step - accuracy: 0.8920 - loss: 0.3120 - val_accuracy: 0.6684 - val_loss: 1.1687
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 142s 182ms/step - accuracy: 0.9134 - loss: 0.2446 - val_accuracy: 0.7805 - val_loss: 0.7267
Epoch 8/10
670/782 ━━━━━━━━━━━━━━━━━━━━ 19s 173ms/step - accuracy: 0.9424 - 

In [ ]:
52-0